Data Ingestion

In [ ]:
# For PurpleAir

from pathlib import Path
import json
import time
import pandas as pd
import requests

checkpoint_dir = Path("Datas/purpleair_checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

empty_log = checkpoint_dir / "empty_requests.json"

if empty_log.exists():
    empty_requests = set(json.loads(empty_log.read_text()))
else:
    empty_requests = set()


In [2]:
import os
import requests
import pandas as pd

import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ["PURPLEAIR_API_KEY"]

url = "https://api.purpleair.com/v1/sensors"

headers = {
    "X-API-Key": API_KEY    
}

url = "https://api.purpleair.com/v1/sensors"

params = {
    "fields": "name,latitude,longitude,location_type,last_seen",
    "nwlng": -74.30,
    "nwlat": 40.92,
    "selng": -73.65,
    "selat": 40.48,
    "location_type": 0
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

response.raise_for_status()

data = response.json()

df_sensors = pd.DataFrame(
    data["data"],
    columns=data["fields"]
)

# The PurpleAir API bounding box is only a preliminary filter.
# NYC borough polygons remove NJ / Nassau / Westchester sensors within that box.
import geopandas as gpd

NYC_BOROUGHS_URL = "https://data.cityofnewyork.us/resource/gthc-hcne.geojson?$limit=10"
boroughs = gpd.read_file(NYC_BOROUGHS_URL).to_crs("EPSG:4326")

# Check sensor metadata BEFORE requesting the expensive historical data.
for col in ["latitude", "longitude"]:
    df_sensors[col] = pd.to_numeric(df_sensors[col], errors="coerce")

df_sensors = df_sensors.dropna(subset=["latitude", "longitude"]).copy()
sensor_points = gpd.GeoDataFrame(
    df_sensors,
    geometry=gpd.points_from_xy(df_sensors["longitude"], df_sensors["latitude"]),
    crs="EPSG:4326"
)

before_count = len(df_sensors)
matched = gpd.sjoin(
    sensor_points,
    boroughs[["boroname", "geometry"]],
    how="inner",
    predicate="intersects"
)
# A point on a shared borough boundary may match twice: retain it once.
df_sensors = pd.DataFrame(
    matched.drop(columns=["geometry", "index_right", "boroname"])
).drop_duplicates(subset=["sensor_index"]).reset_index(drop=True)

print(f"NYC sensors retained: {len(df_sensors)} / {before_count}")
df_sensors.head()


NYC sensors retained: 62 / 90


,sensor_index,last_seen,name,location_type,latitude,longitude
0,4803,1794675387,West 90th Street near CPW,0,40.787937,-73.96807
1,276266,1794678682,Crown St bet Bedford & Rogers Aves,0,40.666515,-73.95539
2,282988,1794678636,a161,0,40.610620,-73.93604
3,284972,1794678678,LESBreathe1,0,40.713257,-73.97851
4,285560,1794678628,Flushing 42,0,40.756435,-73.82577


In [15]:
import requests
import pandas as pd
import time
from datetime import datetime, timezone

# Function that gets sensor data (specified by sensor_id) from time stamp start to end
def get_sensor_history(sensor_id, start, end):
    url = f"https://api.purpleair.com/v1/sensors/{sensor_id}/history"

    params = {
        "start_timestamp": start,
        "end_timestamp": end,
        "average": 1440,
        "fields": "pm2.5_atm_a,pm2.5_atm_b"
    }

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30
    )

    response.raise_for_status()
    history = response.json()

    print("Status:", response.status_code)
    print("Fields:", history.get("fields"))
    print("Sensor rows:", len(history.get("data") or []))

    if not history.get("data"):
        return pd.DataFrame()

    df = pd.DataFrame(
        history["data"],
        columns=history["fields"]
    )

    df["sensor_index"] = sensor_id

    return df

# Get data for each sensor from 2016-2026
for sensor_id in df_sensors["sensor_index"]:
    for year in range(2016, 2027):

        # Define the requested date range
        start = pd.Timestamp(f"{year}-01-01", tz="UTC")
        end = min(
            pd.Timestamp(f"{year + 1}-01-01", tz="UTC"),
            pd.Timestamp.now(tz="UTC")
        )

        if start >= end:
            continue

        # Save the requested end date in the checkpoint name
        # so partial years can be updated later
        request_id = f"{int(sensor_id)}_{year}_{end:%Y%m%d}"
        filepath = checkpoint_dir / f"sensor_{request_id}.csv"

        # Skip completed downloads
        if filepath.exists() or request_id in empty_requests:
            print(f"Skipping completed request: {request_id}")
            continue

        try:
            df_temp = get_sensor_history(
                int(sensor_id),
                start.isoformat(),
                end.isoformat()
            )

            if not df_temp.empty:
                # Save to temporary file first
                temp_path = filepath.with_suffix(".tmp")
                df_temp.to_csv(temp_path, index=False)

                # Rename only after writing succeeds
                temp_path.replace(filepath)

                print(f"Saved: {filepath}")

            else:
                # Record successfully completed empty requests
                empty_requests.add(request_id)

                temp_log = empty_log.with_suffix(".tmp")
                temp_log.write_text(
                    json.dumps(sorted(empty_requests))
                )
                temp_log.replace(empty_log)

                print(f"No data: {request_id}")

        except requests.RequestException as e:
            print(f"Error for {request_id}: {e}")

            # Stop on suspected quota errors
            response = getattr(e, "response", None)

            if response is not None:
                if (
                    response.status_code == 429
                    or "over-quota" in response.url
                ):
                    raise RuntimeError(
                        "PurpleAir quota/rate limit reached. "
                        "Completed downloads are saved."
                    ) from e

            # Other failures are not checkpointed
            # so they can be retried later

        finally:
            time.sleep(1.5)




Status: 200
Fields: ['time_stamp', 'pm2.5_atm_a', 'pm2.5_atm_b']
Sensor rows: 0
No data: 4803_2016_20170101
Status: 200
Fields: ['time_stamp', 'pm2.5_atm_a', 'pm2.5_atm_b']
Sensor rows: 8
Saved: Datas/purpleair_checkpoints/sensor_4803_2017_20180101.csv
Status: 200
Fields: ['time_stamp', 'pm2.5_atm_a', 'pm2.5_atm_b']
Sensor rows: 361
Saved: Datas/purpleair_checkpoints/sensor_4803_2018_20190101.csv
Status: 200
Fields: ['time_stamp', 'pm2.5_atm_a', 'pm2.5_atm_b']
Sensor rows: 29
Saved: Datas/purpleair_checkpoints/sensor_4803_2019_20200101.csv
Status: 200
Fields: ['time_stamp', 'pm2.5_atm_a', 'pm2.5_atm_b']
Sensor rows: 0
No data: 4803_2020_20210101
Status: 200
Fields: ['time_stamp', 'pm2.5_atm_a', 'pm2.5_atm_b']
Sensor rows: 0
No data: 4803_2021_20220101
Status: 200
Fields: ['time_stamp', 'pm2.5_atm_a', 'pm2.5_atm_b']
Sensor rows: 0
No data: 4803_2022_20230101
Status: 200
Fields: ['time_stamp', 'pm2.5_atm_a', 'pm2.5_atm_b']
Sensor rows: 0
No data: 4803_2023_20240101
Status: 200
Fields: ['

In [30]:
len(df_sensors)

62

In [16]:
# Merge the purple-air files together

df_history = pd.concat(
    [
        pd.read_csv(file)
        for file in checkpoint_dir.glob("sensor_*.csv")
    ],
    ignore_index=True
)

In [17]:
# Channel Filtering 
import numpy as np

a = pd.to_numeric(df_history["pm2.5_atm_a"], errors="coerce")
b = pd.to_numeric(df_history["pm2.5_atm_b"], errors="coerce")

# Average of both channels
channel_mean = (a + b) / 2

# Absolute difference
difference = (a - b).abs()

# Agreement threshold
agree = (
    a.notna() &
    b.notna() &
    (a >= 0) &
    (b >= 0) &
    (
        (difference <= 5) |
        (difference <= 0.61 * channel_mean)
    )
)

df_history["channel_agrees"] = agree
df_history["pm25_clean"] = channel_mean.where(agree)


# Only days when both channels produced valid, nonnegative readings
df_history["valid_pair"] = (
    a.notna() & b.notna() &
    (a >= 0) & (b >= 0)
)

sensor_quality = (
    df_history.groupby("sensor_index")
    .agg(
        total_days=("channel_agrees", "size"),
        paired_days=("valid_pair", "sum"),
        agree_days=("channel_agrees", "sum")
    )
    .reset_index()
)

# Fraction of paired days where channels agree
sensor_quality["agreement_rate"] = (
    sensor_quality["agree_days"] /
    sensor_quality["paired_days"].replace(0, np.nan)
)

sensor_quality.sort_values("agreement_rate")

min_agreement = 0.80

reliable_sensors = sensor_quality.loc[
    (sensor_quality["paired_days"] >= 30) &
    (sensor_quality["agreement_rate"] >= min_agreement),
    "sensor_index"
]

df_clean = df_history[
    df_history["sensor_index"].isin(reliable_sensors) &
    df_history["channel_agrees"]
].copy()

In [18]:
df_clean

,time_stamp,pm2.5_atm_a,pm2.5_atm_b,sensor_index,channel_agrees,pm25_clean,valid_pair
0,2026-04-19T00:00:00Z,3.1,4.7,175633,True,3.90,True
1,2026-05-14T00:00:00Z,3.7,5.4,175633,True,4.55,True
2,2026-03-28T00:00:00Z,2.6,3.9,175633,True,3.25,True
3,2026-07-09T00:00:00Z,7.3,10.6,175633,True,8.95,True
4,2026-07-02T00:00:00Z,16.5,23.8,175633,True,20.15,True
...,...,...,...,...,...,...,...
52146,2026-06-29T00:00:00Z,18.1,20.0,285560,True,19.05,True
52147,2026-08-06T00:00:00Z,20.2,20.3,285560,True,20.25,True
52148,2026-04-27T00:00:00Z,7.9,8.3,285560,True,8.10,True
52149,2026-07-21T00:00:00Z,17.9,17.8,285560,True,17.85,True


In [19]:
# We want to merge df_history and df_sensors in order to get latitude and longitude coordinates'
df_clean = df_clean.merge(
    df_sensors[['sensor_index', 'latitude', 'longitude']],
    on='sensor_index',
    how='inner'
)


In [21]:
df_clean["time_stamp"] = pd.to_datetime(
    df_clean["time_stamp"],
    utc=True
)

df_clean["Year"] = df_clean["time_stamp"].dt.year
df_clean["Month"] = df_clean["time_stamp"].dt.month

In [23]:
df_clean = (
    df_clean.groupby(
        ['sensor_index', 'Month', 'Year'],
        dropna=False
    )
    .agg({
        'pm25_clean': 'mean',
        'latitude': 'first',
        'longitude': 'first'
    })
    .reset_index()
)

In [28]:
df_clean.to_csv('purple_air_monthlypm25.csv', index=False)

In [27]:
df_clean

,sensor_index,Month,Year,pm25_clean,latitude,longitude
0,4803,1,2018,14.830645,40.787937,-73.96807
1,4803,1,2019,9.651724,40.787937,-73.96807
2,4803,1,2026,11.040323,40.787937,-73.96807
3,4803,2,2018,15.351786,40.787937,-73.96807
4,4803,2,2026,12.771429,40.787937,-73.96807
...,...,...,...,...,...,...
1465,310160,11,2026,8.182143,40.647360,-73.97419
1466,319136,8,2026,13.294643,40.587710,-73.81614
1467,319136,9,2026,6.608333,40.587710,-73.81614
1468,319136,10,2026,6.985484,40.587710,-73.81614


In [ ]:
df_history['sensor_index'].nunique()

67

In [ ]:
# Air Data

df = pd.read_csv("epa_data/daily_88101_2016.csv")
for i in range(17, 27):
    df= pd.concat([df, pd.read_csv(f'epa_data/daily_88101_20{str(i)}.csv')], ignore_index=True)

df = df[
    (df['County Code'].isin([5, 47, 61, 81, 85])) & 
    (df['State Code'] == 36)
]      
df = df[['County Code', 'Latitude', 'Longitude', 'Date Local', 'Arithmetic Mean', 'Address', 'State Name', 'Site Num']]
df['Date Local'] = pd.to_datetime(df['Date Local'])
df['Month'] = df['Date Local'].dt.month 
df['Year'] = df['Date Local'].dt.year
df = df.drop_duplicates()

df_monthly = (
    df.groupby(
        ['County Code', 'Month', 'Year', 'Site Num'],
        dropna=False
    )
    .agg({
        'Arithmetic Mean': 'mean',
        'Latitude': 'first',
        'Longitude': 'first',
        'Address': 'first',
        'State Name': 'first',
    })
    .reset_index()
)
borough_map = {
    5: 'Bronx',
    47: 'Brooklyn',
    61: 'Manhattan',
    81: 'Queens',
    85: 'Staten Island'
}

df_monthly['Boroughs'] = df_monthly['County Code'].map(borough_map).fillna('Unknown')
df_monthly

/var/folders/yl/_whlcz951v78mbcsqg64rxd00000gn/T/ipykernel_8657/821518165.py:3: DtypeWarning: Columns (13: Event Type) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("epa_data/daily_88101_2016.csv")
/var/folders/yl/_whlcz951v78mbcsqg64rxd00000gn/T/ipykernel_8657/821518165.py:5: DtypeWarning: Columns (13: Event Type, 27: CBSA Name) have mixed types. Specify dtype option on import or set low_memory=False.
  df= pd.concat([df, pd.read_csv(f'epa_data/daily_88101_20{str(i)}.csv')], ignore_index=True)
/var/folders/yl/_whlcz951v78mbcsqg64rxd00000gn/T/ipykernel_8657/821518165.py:5: DtypeWarning: Columns (13: Event Type) have mixed types. Specify dtype option on import or set low_memory=False.
  df= pd.concat([df, pd.read_csv(f'epa_data/daily_88101_20{str(i)}.csv')], ignore_index=True)


In [ ]:
df_monthly
len(df_monthly)

1111